# Time-Frequency-Polarization analysis: tutorial

This tutorial aims at demonstrating different tools available within the `timefrequency` module of `BiSPy`. The examples provided here come along with the paper

* Julien Flamant, Nicolas Le Bihan, Pierre Chainais: “Time-frequency analysis of bivariate signals”, Applied and Computational Harmonic Analysis, 2017; [arXiv:1609.02463](http://arxiv.org/abs/1609.02463), [doi:10.1016/j.acha.2017.05.007](https://doi.org/10.1016/j.acha.2017.05.007).

The paper contains theoretical results and several applications that can be reproduced with the following tutorial.

This page is a Jupyter notebook, run every time the documentation is built, so its outputs come from the current version of BiSPy. [Download the notebook](https://github.com/jflamant/bispy/blob/master/docs/tutorials/timefrequency.ipynb) to run it yourself.

## Load `bispy` and necessary modules

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import quaternion  # load the quaternion module
import bispy as bsp

## Instantaneous polarization: the quaternion embedding

The [getting-started tutorial](gettingstarted.ipynb) writes a monochromatic bivariate signal as the projection of the quaternion $a\,e^{\boldsymbol{i}\theta}e^{-\boldsymbol{k}\chi}e^{\boldsymbol{j}(2\pi f_0 t + \varphi)}$. For an arbitrary bivariate signal $x(t)$, the *quaternion embedding*, or $\mathbb{H}$-embedding, $x_+(t)$ plays the role of the analytic signal: in the QFT domain, it cancels the negative frequencies and doubles the positive ones,

$$X_+(f) = 2X(f) \;\text{ for } f > 0, \qquad X_+(f) = 0 \;\text{ for } f < 0 .$$

The signal is the $1, \boldsymbol{i}$ part of its embedding, and the polar form of the embedding,

$$x_+(t) = a(t)\,e^{\boldsymbol{i}\theta(t)}e^{-\boldsymbol{k}\chi(t)}e^{\boldsymbol{j}\varphi(t)},$$

defines the instantaneous amplitude, orientation, ellipticity and phase. For a bivariate AM-FM signal, whose amplitude and polarization vary slowly compared to its phase, they match the parameters of the model. This section follows the third notebook of the EUSIPCO 2024 tutorial by Nicolas Le Bihan and Julien Flamant, [on GitHub](https://github.com/ricochet-anr/2024_eusipco_tutorial_bivariate_signal_processing).

Let us build such a signal: a linear chirp whose ellipse rotates (the orientation goes from $-\pi/4$ to $3\pi/4$) and changes direction (the ellipticity goes from $\pi/6$ to $-\pi/6$). With `Hembedding=False`, `signals.bivariateAMFM` returns the bivariate signal itself.

In [ ]:
N = 1024  # length of the signal
t = np.linspace(0, 1, N)  # time vector

a = bsp.utils.windows.hanning(N)  # amplitude
theta = -np.pi / 4 + np.pi * t  # rotating orientation
chi = np.pi / 6 - np.pi / 3 * t  # ellipticity changing sign
phi = 2 * np.pi * (50 * t + 50 * t**2)  # linear chirp, from 50 to 150 Hz

x = bsp.signals.bivariateAMFM(a, theta, chi, phi, Hembedding=False)
fig, ax = bsp.utils.visual.plot2D(t, x)

`timefrequency.Hembedding` computes the embedding and its polar form, stored in the attributes `a`, `theta`, `chi` and `phi`. Its method `plot` displays them; here, the parameters of the model are added as dashed lines.

> **Note.** Pass the bivariate signal itself to `Hembedding`, not a quaternion embedding such as the default output of `bivariateAMFM`: the embedding of an embedding has twice its amplitude.

In [ ]:
H = bsp.timefrequency.Hembedding(x)
fig, ax = H.plot(t)

# parameters of the model (the orientation is defined modulo pi)
ax[0].plot(t, a, 'k--')
ax[1].plot(t, (theta + np.pi / 2) % np.pi - np.pi / 2, 'k--')
ax[2].plot(t, chi, 'k--')
ax[3].plot(t, phi, 'k--');

The instantaneous parameters match the model, except near both ends, where the amplitude vanishes and polarization is undefined. The orientation is defined modulo $\pi$: when the major axis of the ellipse passes the vertical, at $\theta = \pi/2$, it continues from $-\pi/2$. `plot` unwraps the phase so that it grows smoothly; with `unwrap=False`, it shows the phase in $(-\pi, \pi]$.

Instantaneous parameters describe a single component. Add a second chirp, at higher frequencies, and the embedding mixes both: its parameters oscillate and describe neither chirp.

In [ ]:
x2 = bsp.signals.bivariateAMFM(a, np.pi / 3, 0, 2 * np.pi * (250 * t + 50 * t**2), Hembedding=False)
fig, ax = bsp.timefrequency.Hembedding(x + x2).plot(t)

Time-frequency representations first separate the components, then give the polarization of each one. This is what the Q-STFT and the Q-CWT below do.

## Quaternion Short-Term Fourier Transform (Q-STFT) example

To illustrate the behaviour of the Q-STFT, we construct a simple signal made of two linear chirps, each having its own instantaneous polarization properties. 

First, define some constants:

In [ ]:
N = 1024 # length of the signal

# linear chirps constants
a = 250*np.pi
b = 50*np.pi
c = 150*np.pi

Then define the instantaneous amplitudes, orientation, ellipticity and phase of each linear chirp. The amplitudes are taken equal - just a Hanning window. 

In [ ]:
# time vector
t = np.linspace(0, 1, N)

# first chirp
theta1 = np.pi/4 # constant orientation
chi1 = np.pi/6-t # reversing ellipticity
phi1 = b*t+a*t**2 # linear chirp

# second chirp
theta2 = np.pi/4*10*t # rotating orientation
chi2 = 0 # constant null ellipticity
phi2 = c*t+a*t**2 # linear chirp

# common amplitude -- simply a window
env = bsp.utils.windows.hanning(N)

We can now construct the two components and sum it. To do so, we use the function `signals.bivariateAMFM` to compute directly the quaternion embeddings of each linear chirp. 

In [ ]:
# define chirps x1 and x2
x1 = bsp.signals.bivariateAMFM(env, theta1, chi1, phi1)
x2 = bsp.signals.bivariateAMFM(env, theta2, chi2, phi2)

# sum it
x = x1 + x2

Let us have a look at the signal `x[t]`

In [ ]:
fig, ax = bsp.utils.visual.plot2D(t, x)

Now we can compute the Q-STFT. First initialize the object Q-STFT

In [ ]:
S = bsp.timefrequency.QSTFT(x, t)

And compute:

In [ ]:
S.compute(window='hamming', nperseg=101, noverlap=100, nfft=N)

Let us have a look at Time-Frequency Stokes parameters S1, S2 and S3

In [ ]:
fig, ax = S.plotStokes()

Alternatively, we can compute the instantaneous polarization properties from the ridges of the Q-STFT.

Extract the ridges:

In [ ]:
S.extractRidges()

And plot (`quivertdecim` controls the time-decimation of the quiver plot, for a cleaner view):

In [ ]:
fig, ax = S.plotRidges(quivertdecim=30)

The two representations are equivalent and provide the same information: time, frequency and polarization properties of the bivariate signal. A direct inspection shows that instantaneous parameters of each components are recovered by both representations. 

## Quaternion Continuous Wavelet Transform (Q-CWT) example

The Q-STFT method has the same limitations as the usual STFT, that is not the ideal tool to analyze signals spanning a wide range of frequencies over short time scales. We revisit here the classic two chirps example in its bivariate (polarized) version. 

As before, let us first define some constants:

In [ ]:
N = 1024 # length of the signal

# hyperbolic chirps parameters
alpha = 15*np.pi
beta = 5*np.pi
tup = 0.8 # set blow-up time value

Now, let us define the instantaneous amplitudes, orientation, ellipticity and phase of each linear chirp. The chirps are also windowed. 

In [ ]:
t = np.linspace(0, 1, N) # time vector

# chirp 1 parameters
theta1 = -np.pi/3 # constant orientation
chi1 = np.pi/6 # constant ellipticity
phi1 = alpha/(.8-t) # hyperbolic chirp

# chirp 2 parameters
theta2 = 5*t # rotating orientation
chi2 = -np.pi/10 # constant ellipticity
phi2 = beta/(.8-t) # hyperbolic chirp

# envelope
env = np.zeros(N)
Nmin = int(0.1*N) # minimum value of N such that x is nonzero
Nmax = int(0.75*N) # maximum value of N such that x is nonzero

env[Nmin:Nmax] = bsp.utils.windows.hanning(Nmax-Nmin)

Construct the two components and sum it. Again we use the function `utils.bivariateAMFM` to compute directly the quaternion embeddings of each linear chirp.

In [ ]:
x1  = bsp.signals.bivariateAMFM(env, theta1, chi1, phi1)
x2  = bsp.signals.bivariateAMFM(env, theta2, chi2, phi2)

x = x1 + x2

Let us visualize the resulting signal, `x[t]`

In [ ]:
fig, ax = bsp.utils.visual.plot2D(t, x)

Now, we can compute its Q-CWT. First define the wavelet parameters and initialize the QCWT object:

In [ ]:
waveletParams = dict(type='Morse', beta=12, gamma=3)
S = bsp.timefrequency.QCWT(x, t)

And compute:

In [ ]:
fmin = 0.01
fmax = 400
S.compute(fmin, fmax, waveletParams, N)

Let us have a look at Time-Scale Stokes parameters S1, S2 and S3

In [ ]:
fig, ax = S.plotStokes()

Similarly we can compute the instantaneous polarization attributes from the ridges of the Q-CWT.

In [ ]:
S.extractRidges()

And plot the results

In [ ]:
fig, ax = S.plotRidges(quivertdecim=40)

Again, both representations are equivalent and provide the same information: time, scale and polarization properties of the bivariate signal. A direct inspection shows that instantaneous parameters of each components are recovered by both representations. 